In [1]:
import socket
print(socket.gethostname())

jupyter-duddukuntak1


In [2]:
# import os

# os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/torchinductor"
# os.environ["TORCH_COMPILE_CACHE_DIR"] = "/tmp/torchcompile"
# os.environ["XDG_CACHE_HOME"] = "/tmp"
# os.environ["HOME"] = "/home/jovyan"

In [2]:
import sys
print(sys.executable)

/home/jovyan/yolov8_env/bin/python


In [3]:
!nvidia-smi

Mon Mar  2 15:00:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:1C:00.0 Off |                    0 |
| N/A   32C    P0            118W /  600W |     554MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.10.0+cu128
CUDA available: True


In [5]:

from IPython.display import Image
import ultralytics 
ultralytics.checks()

Ultralytics 8.4.18 🚀 Python-3.11.10 torch-2.10.0+cu128 CUDA:0 (NVIDIA H200, 143167MiB)
Setup complete ✅ (192 CPUs, 1007.5 GB RAM, 3871.1/7151.8 GB disk)


In [ ]:
# ── Install dependencies (run this cell first) ───────────────────────
# !pip install ultralytics albumentations opencv-python pyyaml -q

import os
import cv2
import random
import shutil
import yaml
import numpy as np
from pathlib import Path
import albumentations as A
from ultralytics import YOLO

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────

#  FIX 1: Correct home dir — confirmed from traceback: /home/jovyan/
HOME_DIR        = Path("/home/jovyan")
BASE_DIR        = HOME_DIR / "pole_data" / "pole_data"

TRAIN_IMG_DIR   = BASE_DIR / "train" / "images"
TRAIN_LBL_DIR   = BASE_DIR / "train" / "labels"
VALID_IMG_DIR   = BASE_DIR / "valid" / "images"
VALID_LBL_DIR   = BASE_DIR / "valid" / "labels"
TEST_IMG_DIR    = BASE_DIR / "test"  / "images"
TEST_LBL_DIR    = BASE_DIR / "test"  / "labels"

# Albumentations output (augmented training images)
AUG_TRAIN_IMG   = BASE_DIR / "train_augmented" / "images"
AUG_TRAIN_LBL   = BASE_DIR / "train_augmented" / "labels"

# data.yaml
ORIGINAL_YAML   = BASE_DIR / "data.yaml"
AUG_YAML        = BASE_DIR / "data_augmented.yaml"

# YOLOv8 settings —  Optimized for 4x NVIDIA H200 (143GB VRAM each)
MODEL_WEIGHTS   = "yolov8x.pt"                         #  UPGRADED: largest model — H200 handles it easily
PROJECT_NAME    = str(HOME_DIR / "gsv_pole_detection") # → /home/jovyan/gsv_pole_detection
RUN_NAME        = "exp1"
EPOCHS          = 100
IMG_SIZE        = 640                  #  FASTER: 4x speedup vs 1280
BATCH_SIZE      = 32                  #  FIXED: single H200 — 32 is optimal
DEVICE          = 0                   #  FIXED: only 1 GPU visible (CUDA_VISIBLE_DEVICES=0)
WORKERS         = 1                #  FIXED: server has restricted /dev/shm — must use 0
AUG_COPIES      = 1                   # 2 augmented copies → 3x dataset size


# ─────────────────────────────────────────────────────────────────────
# STEP 0 — Auto-fix data.yaml to use absolute paths
# ─────────────────────────────────────────────────────────────────────

def fix_data_yaml():
    if not ORIGINAL_YAML.exists():
        raise FileNotFoundError(f"data.yaml not found at: {ORIGINAL_YAML}")

    with open(ORIGINAL_YAML, "r") as f:
        data = yaml.safe_load(f)

    data["train"] = str(TRAIN_IMG_DIR)
    data["val"]   = str(VALID_IMG_DIR)
    data["test"]  = str(TEST_IMG_DIR)

    with open(ORIGINAL_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)

    print(f"[Step 0] data.yaml fixed with absolute paths:")
    print(f"  train → {TRAIN_IMG_DIR}")
    print(f"  val   → {VALID_IMG_DIR}")
    print(f"  test  → {TEST_IMG_DIR}")


# ─────────────────────────────────────────────────────────────────────
# STEP 1 — ALBUMENTATIONS PIPELINE (compatible with installed version)
# ─────────────────────────────────────────────────────────────────────

def get_albumentations_pipeline():
    """
     FIX 3: RandomSunFlare — removed invalid 'angle_upper',
              'num_flare_circles_range', 'src_color' params.
              Using only safe, version-agnostic parameters.
    """
    return A.Compose(
        [
            # ── Lighting & Contrast ───────────────────────────────────
            A.RandomBrightnessContrast(p=0.5),
            A.CLAHE(clip_limit=4.0, p=0.3),

            # ── Noise ─────────────────────────────────────────────────
            A.GaussNoise(noise_scale_factor=0.1, p=0.3),

            # ── Blur (vehicle motion + out-of-focus distant poles) ────
            A.MotionBlur(blur_limit=5, p=0.2),
            A.GaussianBlur(blur_limit=3, p=0.2),

            # ── Weather Effects ───────────────────────────────────────
            A.RandomFog(
                fog_coef_range=(0.1, 0.3),
                alpha_coef=0.1,
                p=0.15
            ),
            A.RandomRain(
                slant_range=(-10, 10),
                drop_length=10,
                drop_width=1,
                brightness_coefficient=0.9,
                p=0.1
            ),
            #  FIX 3: Only use flare_roi — safe across all versions
            A.RandomSunFlare(
                flare_roi=(0, 0, 1, 0.5),  # Upper half of image only
                p=0.1
            ),
            A.RandomShadow(
                shadow_roi=(0, 0.5, 1, 1),  # Lower half (building shadows)
                num_shadows_limit=(1, 2),
                shadow_dimension=5,
                p=0.2
            ),
        ],
        bbox_params=A.BboxParams(
            format="yolo",
            label_fields=["class_labels"],
            min_visibility=0.3,
        ),
    )


# ─────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────

def read_yolo_labels(label_path):
    bboxes, class_labels = [], []
    if not os.path.exists(label_path):
        return bboxes, class_labels
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls = int(parts[0])
                cx, cy, w, h = map(float, parts[1:])
                bboxes.append([cx, cy, w, h])
                class_labels.append(cls)
    return bboxes, class_labels


def write_yolo_labels(label_path, bboxes, class_labels):
    with open(label_path, "w") as f:
        for cls, bbox in zip(class_labels, bboxes):
            f.write(f"{cls} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")


# ─────────────────────────────────────────────────────────────────────
# STEP 1 — Apply Albumentations to training set
# ─────────────────────────────────────────────────────────────────────

def apply_albumentations_to_dataset(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    os.makedirs(AUG_TRAIN_IMG, exist_ok=True)
    os.makedirs(AUG_TRAIN_LBL, exist_ok=True)

    pipeline    = get_albumentations_pipeline()
    image_paths = (
        sorted(Path(TRAIN_IMG_DIR).glob("*.jpg"))
        + sorted(Path(TRAIN_IMG_DIR).glob("*.jpeg"))
        + sorted(Path(TRAIN_IMG_DIR).glob("*.png"))
    )

    print(f"\n[Step 1] Found {len(image_paths)} training images")
    print(f"         Generating {AUG_COPIES} augmented copies each "
          f"→ {len(image_paths) * (AUG_COPIES + 1)} total images\n")

    failed = 0

    for idx, img_path in enumerate(image_paths):
        img_path = Path(img_path)
        stem     = img_path.stem
        ext      = img_path.suffix
        lbl_path = TRAIN_LBL_DIR / (stem + ".txt")

        bboxes, class_labels = read_yolo_labels(str(lbl_path))

        image = cv2.imread(str(img_path))
        if image is None:
            print(f"  [WARN] Could not read {img_path.name}, skipping.")
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Save original into augmented folder
        cv2.imwrite(
            str(AUG_TRAIN_IMG / (stem + ext)),
            cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        )
        if lbl_path.exists():
            shutil.copy(str(lbl_path), str(AUG_TRAIN_LBL / (stem + ".txt")))
        else:
            open(str(AUG_TRAIN_LBL / (stem + ".txt")), "w").close()

        # Save augmented copies
        for i in range(AUG_COPIES):
            try:
                result   = pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
                aug_stem = f"{stem}_aug{i + 1}"
                cv2.imwrite(
                    str(AUG_TRAIN_IMG / (aug_stem + ext)),
                    cv2.cvtColor(result["image"], cv2.COLOR_RGB2BGR)
                )
                write_yolo_labels(
                    str(AUG_TRAIN_LBL / (aug_stem + ".txt")),
                    result["bboxes"],
                    result["class_labels"]
                )
            except Exception as e:
                failed += 1
                print(f"  [WARN] Augmentation failed for {stem} copy {i+1}: {e}")

        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(image_paths)} images...")

    total = len(list(AUG_TRAIN_IMG.glob("*.*")))
    print(f"\n  Done. Total images in train_augmented/images: {total}")
    if failed > 0:
        print(f"  [WARN] {failed} augmentation(s) failed — check warnings above.")
    else:
        print(f"  All augmentations completed successfully ")


# ─────────────────────────────────────────────────────────────────────
# STEP 2 — Create data_augmented.yaml
# ─────────────────────────────────────────────────────────────────────

def create_augmented_yaml():
    with open(ORIGINAL_YAML, "r") as f:
        data = yaml.safe_load(f)

    data["train"] = str(AUG_TRAIN_IMG)
    data["val"]   = str(VALID_IMG_DIR)
    data["test"]  = str(TEST_IMG_DIR)

    with open(AUG_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)

    print(f"\n[Step 2] data_augmented.yaml created at:\n  {AUG_YAML}")
    print(f"  train → {AUG_TRAIN_IMG}")
    print(f"  val   → {VALID_IMG_DIR}")
    print(f"  test  → {TEST_IMG_DIR}")


# ─────────────────────────────────────────────────────────────────────
# STEP 3 — YOLOv8 Training
# ─────────────────────────────────────────────────────────────────────

def train_yolov8():
    model = YOLO(MODEL_WEIGHTS)

    print(f"\n[Step 3] Starting YOLOv8 training...")
    print(f"  Model    : {MODEL_WEIGHTS}")
    print(f"  Epochs   : {EPOCHS}")
    print(f"  Img size : {IMG_SIZE}")
    print(f"  Batch    : {BATCH_SIZE}")
    print(f"  Device   : {DEVICE}")
    print(f"  Workers  : {WORKERS}\n")

    results = model.train(
        data=str(AUG_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        project=PROJECT_NAME,
        name=RUN_NAME,
        exist_ok=True,

        # ── AMP (Automatic Mixed Precision) ──────────────────────────
        amp=True,           #  H200 supports BF16 — faster training, same accuracy

        # ── HSV Color Jitter ─────────────────────────────────────────
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.5,

        # ── Geometric Augmentations ──────────────────────────────────
        degrees=4.0,
        translate=0.1,
        scale=0.5,
        shear=1.5,
        perspective=0.0003,

        # ── Flip ─────────────────────────────────────────────────────
        flipud=0.0,         # DISABLED — GSV always upright
        fliplr=0.5,         # Poles appear on both sides of road

        # ── Composite Augmentations ──────────────────────────────────
        mosaic=1.0,
        mixup=0.12,
        copy_paste=0.25,

        # ── Occlusion ────────────────────────────────────────────────
        erasing=0.25,

        # ── Mosaic Scheduling ────────────────────────────────────────
        close_mosaic=10,

        # ── Optimizer — scaled LR for large batch ────────────────────
        optimizer="AdamW",
        lr0=0.002,          #  SCALED UP: linear scaling rule (0.001 × batch64/batch16 × 0.5)
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=5,    #  INCREASED: longer warmup for large batch + multi-GPU
        warmup_momentum=0.8,

        # ── Output ───────────────────────────────────────────────────
        save=True,
        save_period=10,
        plots=True,
        verbose=True,
        val=True,
        patience=20,
    )
    return results


# ─────────────────────────────────────────────────────────────────────
# STEP 4 — Evaluate on Test Set
# ─────────────────────────────────────────────────────────────────────

def evaluate_model():
    best_weights = f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"
    if not os.path.exists(best_weights):
        print(f"\n[WARN] best.pt not found at {best_weights}. Skipping evaluation.")
        return

    print(f"\n[Step 4] Evaluating on test set...")
    model   = YOLO(best_weights)
    metrics = model.val(
        data=str(AUG_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,           #  Also 0 here for consistency
        project=PROJECT_NAME,
        name=f"{RUN_NAME}_test_eval",
        verbose=True,
    )

    print("\n  ── Test Set Results ──────────────────────────────")
    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")
    print("  ──────────────────────────────────────────────────")
    return metrics


# ─────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 62)
    print("  YOLOv8 Pole & Wire Detection — JupyterLab Training Pipeline")
    print("=" * 62)

    # ── Verify all required folders exist ─────────────────────────────
    required = {
        "train/images" : TRAIN_IMG_DIR,
        "train/labels" : TRAIN_LBL_DIR,
        "valid/images" : VALID_IMG_DIR,
        "valid/labels" : VALID_LBL_DIR,
        "test/images"  : TEST_IMG_DIR,
        "test/labels"  : TEST_LBL_DIR,
    }
    all_ok = True
    print()
    for name, path in required.items():
        exists = path.exists()
        print(f"  [{'OK' if exists else 'MISSING'}] {name}")
        print(f"        → {path}")
        if not exists:
            all_ok = False

    if not all_ok:
        raise FileNotFoundError(
            "\n[ERROR] One or more folders are missing.\n"
            "Expected structure:\n"
            "  /home/jovyan/pole_data/pole_data/\n"
            "  ├── train/\n"
            "  │   ├── images/\n"
            "  │   └── labels/\n"
            "  ├── valid/\n"
            "  │   ├── images/\n"
            "  │   └── labels/\n"
            "  ├── test/\n"
            "  │   ├── images/\n"
            "  │   └── labels/\n"
            "  └── data.yaml\n"
        )

    print()
    fix_data_yaml()                     # Step 0 — Fix relative paths in data.yaml
    apply_albumentations_to_dataset()   # Step 1 — Albumentations pre-processing
    create_augmented_yaml()             # Step 2 — Create data_augmented.yaml
    train_yolov8()                      # Step 3 — YOLOv8 training
    evaluate_model()                    # Step 4 — Test set evaluation

    print("\n  ═══════════════════════════════════════════════════")
    print("  All done! Your model is ready.")
    print("  ═══════════════════════════════════════════════════")
    print(f"  Best weights : {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
    print(f"  Last weights : {PROJECT_NAME}/{RUN_NAME}/weights/last.pt")
    print(f"  Plots/results: {PROJECT_NAME}/{RUN_NAME}/")
    print("\n  To use your model:")
    print("    from ultralytics import YOLO")
    print(f"    model = YOLO('{PROJECT_NAME}/{RUN_NAME}/weights/best.pt')")
    print("    results = model.predict('image.jpg')")

  YOLOv8 Pole & Wire Detection — JupyterLab Training Pipeline

  [OK] train/images
        → /home/jovyan/pole_data/pole_data/train/images
  [OK] train/labels
        → /home/jovyan/pole_data/pole_data/train/labels
  [OK] valid/images
        → /home/jovyan/pole_data/pole_data/valid/images
  [OK] valid/labels
        → /home/jovyan/pole_data/pole_data/valid/labels
  [OK] test/images
        → /home/jovyan/pole_data/pole_data/test/images
  [OK] test/labels
        → /home/jovyan/pole_data/pole_data/test/labels

[Step 0] data.yaml fixed with absolute paths:
  train → /home/jovyan/pole_data/pole_data/train/images
  val   → /home/jovyan/pole_data/pole_data/valid/images
  test  → /home/jovyan/pole_data/pole_data/test/images


/home/jovyan/yolov8_env/lib/python3.11/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()



[Step 1] Found 5522 training images
         Generating 1 augmented copies each → 11044 total images

  Processed 50/5522 images...
  Processed 100/5522 images...
  Processed 150/5522 images...
  Processed 200/5522 images...
  Processed 250/5522 images...
  Processed 300/5522 images...
  Processed 350/5522 images...
  Processed 400/5522 images...
  Processed 450/5522 images...
  Processed 500/5522 images...
  Processed 550/5522 images...
  Processed 600/5522 images...
  Processed 650/5522 images...
  Processed 700/5522 images...
  Processed 750/5522 images...
  Processed 800/5522 images...
  Processed 850/5522 images...
  Processed 900/5522 images...
  Processed 950/5522 images...
  Processed 1000/5522 images...
  Processed 1050/5522 images...
  Processed 1100/5522 images...
  Processed 1150/5522 images...
  Processed 1200/5522 images...
  Processed 1250/5522 images...
  Processed 1300/5522 images...
  Processed 1350/5522 images...
  Processed 1400/5522 images...
  Processed 1450/5522

Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/multiprocessing/queues.py", line 244, in _feed
    obj = _ForkingPickler.dumps(obj)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/queues.py", line 244, in _feed
    obj = _ForkingPickler.dumps(obj)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/reduction.py", line 51, in dumps
    cls(buf, protocol).dump(obj)
  File "/opt/conda/lib/python3.11/multiprocessing/reduction.py", line 51, in dumps
    cls(buf, protocol).dump(obj)
  File "/home/jovyan/yolov8_env/lib/python3.11/site-packages/torch/multiprocessing/reductions.py", line 615, in reduce_storage
    fd, size = storage._share_fd_cpu_()
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/yolov8_env/lib/python3.11/site-packages/torch/storage.py", line 449, in wrapper
    return fn(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File

Image sizes 640 train, 640 val
Using 1 dataloader workers
Logging results to /home/jovyan/gsv_pole_detection/exp1
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      1/100      21.9G      2.938      4.761      2.789         64        640: 0% ──────────── 0/551  0.2s

Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/multiprocessing/queues.py", line 244, in _feed
    obj = _ForkingPickler.dumps(obj)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/queues.py", line 244, in _feed
    obj = _ForkingPickler.dumps(obj)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/reduction.py", line 51, in dumps
    cls(buf, protocol).dump(obj)
  File "/opt/conda/lib/python3.11/multiprocessing/reduction.py", line 51, in dumps
    cls(buf, protocol).dump(obj)
  File "/home/jovyan/yolov8_env/lib/python3.11/site-packages/torch/multiprocessing/reductions.py", line 615, in reduce_storage
    fd, size = storage._share_fd_cpu_()
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/yolov8_env/lib/python3.11/site-packages/torch/multiprocessing/reductions.py", line 615, in reduce_storage
    fd, size = storage._share_fd_cpu_()
              

      1/100      21.9G      2.877      4.598      2.764         63        640: 0% ──────────── 1/551 1.2s/it 0.6s<10:57

Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/multiprocessing/queues.py", line 244, in _feed
    obj = _ForkingPickler.dumps(obj)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/reduction.py", line 51, in dumps
    cls(buf, protocol).dump(obj)
  File "/home/jovyan/yolov8_env/lib/python3.11/site-packages/torch/multiprocessing/reductions.py", line 615, in reduce_storage
    fd, size = storage._share_fd_cpu_()
               ^^^^^^^^^^^^^^^^^^^^^^^^
Traceback (most recent call last):
  File "/home/jovyan/yolov8_env/lib/python3.11/site-packages/torch/storage.py", line 449, in wrapper
    return fn(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/yolov8_env/lib/python3.11/site-packages/torch/storage.py", line 524, in _share_fd_cpu_
    return super()._share_fd_cpu_(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: unable to allocate shared memory(shm) for file </torc

      1/100      21.9G      1.755      1.663       1.72         75        640: 82% ━━━━━━━━━╸── 450/551 2.9it/s 2:37<35.1s